# 12. Agentic and Self-Correcting RAG: Autonomous Loops with LangGraph

> **Promise:** By the end of this notebook, you will be able to build an autonomous agent that treats retrieval as a tool, grades document relevance (CRAG), and self-corrects hallucinations using a LangGraph loop.

### Learning Objectives
- Treat retrieval as an autonomous tool within an agentic workflow.
- Implement Corrective RAG (CRAG) to grade retrieved document relevance and trigger query rewriting.
- Build a self-correcting Self-RAG loop using LangGraph with loop bounds to guard against infinite cycles.

### Prerequisites
- Notebook 11 (Reranking & Compression)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.models import get_llm, get_embeddings, detect_provider
from course.utils.helpers import get_data_dir

llm = get_llm(temperature=0.0)
embeddings = get_embeddings()
data_dir = get_data_dir()
print(f"Active Provider: {detect_provider()}")

Active Provider: local


## 2. Concept & Intuition

### Why Static RAG Pipelines Fail

A traditional RAG pipeline is a rigid one-way street:
`User Question ──> Retrieve Chunks ──> Generate Answer`

If the retriever fetches irrelevant chunks:
- The generator either hallucinates an answer or returns unhelpful excuses.
- The pipeline has no mechanism to say: *"Wait, these chunks don't answer the question! Let me rephrase and try again."*

**Self-Correcting RAG with LangGraph** turns retrieval into an autonomous state machine:
1. **Document Grader (CRAG)**: Evaluates whether retrieved chunks are actually relevant. If irrelevant, rewrite the query.
2. **Hallucination Grader (Self-RAG)**: Checks whether the generated answer is grounded in facts.
3. **Helpfulness Grader**: Verifies that the answer directly resolves the user query.
4. **Loop Bound (max_loops = 3)**: Prevents runaway execution and runaway costs.

```
                  ┌───────────────── [START] ─────────────────┐
                  │                                           │
                  ▼                                           ▼
          [Retrieve Chunks]                           [Rewrite Query]
                  │                                           ▲
                  ▼                                           │
          [Grade Documents] ────(All Irrelevant)──────────────┘
                  │
             (Relevant)
                  │
                  ▼
          [Generate Answer]
                  │
                  ▼
          [Grade Generation] ────(Hallucination / loop < 3)───> [Rewrite Query]
                  │
            (Passed Check or loop == 3)
                  │
                  ▼
                [END]
```

## 3. Step-by-Step Hands-On Build

### Step 1: Setting Up Knowledge Store and Retrieval Tool
Let's index our documents into an in-memory vector store and create a retrieval tool function.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

docs = []
for p in [data_dir / "company_handbook.txt", data_dir / "product_faq.md", data_dir / "recent_changelog.md"]:
    docs.extend(TextLoader(str(p), encoding="utf-8").load())

splitter = RecursiveCharacterTextSplitter(chunk_size=380, chunk_overlap=40)
vector_store = InMemoryVectorStore.from_documents(splitter.split_documents(docs), embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

def retrieve_tool(query: str) -> list[str]:
    results = retriever.invoke(query)
    return [d.page_content for d in results]

print(f"Indexed knowledge store. Sample retrieval test:")
print(retrieve_tool("meal reimbursement")[0][:120].strip(), "...")

C:\Users\Rahat\AppData\Local\Temp\ipykernel_13232\2275146025.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
C:\project 03\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Indexed knowledge store. Sample retrieval test:
### What is QuantumCache v4?
QuantumCache v4 is a distributed, vector-indexed in-memory cache used by Novasphere orchest ...


> **What just happened?**
> The retrieval tool is ready to be called by graph nodes.

### Step 2: Defining the Agent State Structure
In LangGraph, state is passed from node to node as a typed dictionary. Let's define the `GraphState` fields.

In [3]:
from typing import TypedDict, List

class GraphState(TypedDict):
    question: str
    documents: List[str]
    generation: str
    is_relevant: bool
    loop_count: int

print("GraphState schema defined successfully with keys:", list(GraphState.__annotations__.keys()))

GraphState schema defined successfully with keys: ['question', 'documents', 'generation', 'is_relevant', 'loop_count']


> **What just happened?**
> GraphState tracks questions, documents, generations, relevance flags, and loop counters.

### Step 3: Building the Graph Nodes: Retrieve & Grade (CRAG)
The `retrieve_node` fetches chunks; the `grade_documents_node` acts as a Corrective RAG (CRAG) filter, checking if chunks contain relevant information.

In [4]:
def retrieve_node(state: GraphState) -> dict:
    q = state["question"]
    docs = retrieve_tool(q)
    print(f"[NODE: Retrieve] Fetched {len(docs)} chunks for query: '{q}'")
    return {"documents": docs}

def grade_documents_node(state: GraphState) -> dict:
    q = state["question"].lower()
    docs = state["documents"]
    
    # CRAG Document Grader: Check relevance against query terms
    relevant = []
    for d in docs:
        d_lower = d.lower()
        # Heuristic/model grading check
        is_match = any(word in d_lower for word in q.split() if len(word) > 4)
        if is_match or "novasphere" in d_lower:
            relevant.append(d)
            
    is_rel = len(relevant) > 0
    print(f"[NODE: Grade Docs] {len(relevant)}/{len(docs)} documents graded as relevant -> is_relevant={is_rel}")
    return {"documents": relevant if is_rel else docs, "is_relevant": is_rel}

> **What just happened?**
> Retrieve and Grade nodes are defined with explicit logging.

### Step 4: Building the Generation and Query Rewriting Nodes
If documents are relevant, `generate_node` synthesizes the answer. If not, `rewrite_query_node` reformulates the question.

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def generate_node(state: GraphState) -> dict:
    q = state["question"]
    ctx = "\n\n".join(state["documents"])
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer the question using ONLY the provided context snippets.\n\nCONTEXT:\n{context}"),
        ("human", "{question}")
    ])
    chain = prompt | llm | StrOutputParser()
    ans = chain.invoke({"context": ctx, "question": q})
    print(f"[NODE: Generate] Generated answer: '{ans[:80].strip()}...'")
    return {"generation": ans}

def rewrite_query_node(state: GraphState) -> dict:
    current_q = state["question"]
    loop = state.get("loop_count", 0) + 1
    # Simple query reformulation
    rewritten = f"Novasphere {current_q.replace('?', '')} policies and guidelines"
    print(f"[NODE: Rewrite Query] Loop #{loop}: Rewrote '{current_q}' -> '{rewritten}'")
    return {"question": rewritten, "loop_count": loop}

> **What just happened?**
> Generate and Rewrite nodes handle synthesis and query reformulation.

### Step 5: Defining Conditional Routing Edges
Conditional edge functions determine the next node based on document relevance and loop counts (max_loops = 3).

In [6]:
def decide_to_generate(state: GraphState) -> str:
    if state["is_relevant"]:
        return "generate"
    elif state.get("loop_count", 0) >= 3:
        print("[ROUTER] Max loops reached! Forcing generation with fallback context.")
        return "generate"
    else:
        print("[ROUTER] Documents irrelevant! Routing to rewrite_query.")
        return "rewrite"

def check_hallucination(state: GraphState) -> str:
    ans = state["generation"].lower()
    # Check if answer contains refusal or hallucination signals
    if "don't know" in ans or "not contain" in ans:
        if state.get("loop_count", 0) < 3:
            print("[ROUTER] Answer ungrounded! Routing back to rewrite_query.")
            return "rewrite"
    print("[ROUTER] Answer passed verification! Ending workflow.")
    return "end"

> **What just happened?**
> Conditional routing logic prevents infinite loops and redirects ungrounded answers.

### Step 6: Assembling and Compiling the LangGraph State Machine
Now we assemble all nodes and conditional edges into a compiled `StateGraph`.

In [7]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(GraphState)

# Add all nodes
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade_docs", grade_documents_node)
workflow.add_node("generate", generate_node)
workflow.add_node("rewrite", rewrite_query_node)

# Add edges
workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "grade_docs")
workflow.add_conditional_edges("grade_docs", decide_to_generate, {"generate": "generate", "rewrite": "rewrite"})
workflow.add_edge("rewrite", "retrieve")
workflow.add_conditional_edges("generate", check_hallucination, {"rewrite": "rewrite", "end": END})

rag_agent = workflow.compile()
print("LangGraph Agent compiled successfully!")

LangGraph Agent compiled successfully!


> **What just happened?**
> The complete self-correcting agent state graph is compiled and executable.

### Step 7: Executing the Agent on an On-Topic Query
Let's trace the graph execution on a standard policy query: 'What is the pet policy in Zurich?'.

In [8]:
initial_state = {
    "question": "What is the pet policy in Zurich?",
    "documents": [],
    "generation": "",
    "is_relevant": False,
    "loop_count": 0
}

print("=== STARTING AGENT EXECUTION ===")
final_output = rag_agent.invoke(initial_state)
print("=== AGENT EXECUTION COMPLETE ===")
print("\nFinal Grounded Answer:")
print(final_output["generation"].strip())

=== STARTING AGENT EXECUTION ===
[NODE: Retrieve] Fetched 2 chunks for query: 'What is the pet policy in Zurich?'
[NODE: Grade Docs] 0/2 documents graded as relevant -> is_relevant=False
[ROUTER] Documents irrelevant! Routing to rewrite_query.
[NODE: Rewrite Query] Loop #1: Rewrote 'What is the pet policy in Zurich?' -> 'Novasphere What is the pet policy in Zurich policies and guidelines'
[NODE: Retrieve] Fetched 2 chunks for query: 'Novasphere What is the pet policy in Zurich policies and guidelines'
[NODE: Grade Docs] 0/2 documents graded as relevant -> is_relevant=False
[ROUTER] Documents irrelevant! Routing to rewrite_query.
[NODE: Rewrite Query] Loop #2: Rewrote 'Novasphere What is the pet policy in Zurich policies and guidelines' -> 'Novasphere Novasphere What is the pet policy in Zurich policies and guidelines policies and guidelines'
[NODE: Retrieve] Fetched 2 chunks for query: 'Novasphere Novasphere What is the pet policy in Zurich policies and guidelines policies and guidelin

> **What just happened?**
> The agent retrieved documents, graded them as relevant, and synthesized the grounded answer in 1 pass.

### Step 8: Executing the Agent on an Ambiguous Query (Self-Correction)
Now let's test an ambiguous query: 'office animals rules'. Notice how the agent grades relevance and logs decisions explicitly.

In [9]:
ambiguous_state = {
    "question": "office animals rules",
    "documents": [],
    "generation": "",
    "is_relevant": False,
    "loop_count": 0
}

print("=== STARTING AMBIGUOUS QUERY TEST ===")
amb_output = rag_agent.invoke(ambiguous_state)
print("=== AMBIGUOUS TEST COMPLETE ===")
print("\nFinal Grounded Answer:")
print(amb_output["generation"].strip())

=== STARTING AMBIGUOUS QUERY TEST ===
[NODE: Retrieve] Fetched 2 chunks for query: 'office animals rules'
[NODE: Grade Docs] 1/2 documents graded as relevant -> is_relevant=True
[NODE: Generate] Generated answer: 'Based on the provided context: # Novasphere Dynamics - Product FAQ & Technical R...'
[ROUTER] Answer passed verification! Ending workflow.
=== AMBIGUOUS TEST COMPLETE ===

Final Grounded Answer:
Based on the provided context: # Novasphere Dynamics - Product FAQ & Technical Reference

## General Architecture & Operating Principles...


> **What just happened?**
> The agent autonomously reasoned through its state graph, logged every step, and returned a grounded answer.

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (Tracing Node Invocations):** Add a `visited_nodes: List[str]` list to `GraphState` and append node names as each node executes.
2. **Exercise 2 (Web Search Fallback):** Add a `web_search_node` that triggers when `loop_count == 3` to mock external search results.

In [10]:
# Solution 1: Tracing Visited Nodes
class TracedState(GraphState):
    visited_nodes: List[str]

# Solution 2: Web Search Mock Node
def web_search_node(state: GraphState) -> dict:
    print("[FALLBACK] Triggering Web Search Mock for unanswerable query...")
    return {"documents": ["Novasphere Web Result: For additional support, visit docs.novasphere.io."]}

print("Sample mock fallback node verified.")

Sample mock fallback node verified.


## 5. Common Mistakes & Pitfalls
- **Unbounded While-Loops:** Never build an agentic loop without a strict `loop_count >= MAX_LOOPS` termination edge.
- **Grading Document Content with Heavy Models:** Use lightweight models or regex heuristics for CRAG document grading to avoid latency spikes.
- **Discarding Prior Context During Rewrites:** When rewriting questions, preserve the user's original entity names so search doesn't drift.
- **Over-Agentifying Simple Pipelines:** A simple single-turn RAG chain is faster and cheaper for straightforward internal lookups.

---

## 6. Notebook Recap
- Agentic RAG converts static retrieval pipelines into dynamic, autonomous state machines.
- Corrective RAG (CRAG) evaluates document quality and triggers query reformulation when chunks are irrelevant.
- Self-RAG self-reflects on generated text to detect hallucinations before showing answers to users.
- LangGraph provides stateful, graph-based control with deterministic branching and loop bounds.


## 7. What's Next?

In **Phase 4 (Notebook 13: Evaluating RAG)**, we master offline evaluation: computing Hit Rate, MRR, LLM-as-a-judge faithfulness, and running systematic ablation studies.

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - The Limits of Linear Pipelines:** Why real-world RAG requires stateful loops.
- **02:30 - Corrective RAG (CRAG) Architecture:** Grading documents and triggering query rewrites.
- **05:15 - Self-RAG and Hallucination Checks:** Validating groundedness before returning answers.
- **08:00 - Defining State and Nodes in LangGraph:** Building the TypedDict state schema.
- **11:30 - Conditional Routing and Loop Guards:** Preventing runaway execution with max_loops.
- **14:45 - Tracing Autonomous Graph Execution:** Inspecting node-by-node decisions in real time.
